In [74]:
import pandas as pd
import zipfile
import os
import numpy as np
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV, ElasticNetCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, root_mean_squared_error

import os
print(os.getcwd())

/Users/hayleytice/SIADS_696_F26_Team10


In [76]:
try:
    aqi_df = pd.read_csv('Datasets/AQI/AQI_agg.csv')
except:
    aqi_df = pd.read_csv('../Datasets/AQI/AQI_agg.csv')
print(f'AQI dataframe length is: {len(aqi_df)}')
aqi_df.describe()

AQI dataframe length is: 2286


,Year,Median AQI
count,2286.000000,2286.000000
mean,2002.079615,40.128806
std,12.953029,7.197556
min,1980.000000,13.881630
25%,1991.000000,35.657875
50%,2002.000000,40.193498
75%,2013.000000,44.411370
max,2024.000000,71.000000


In [3]:
seds_df = pd.read_csv('Datasets/SEDS/df_seds_state_clusters.csv')
print(f'SEDS dataframe length is: {len(seds_df)}')
seds_df.describe()

SEDS dataframe length is: 3315


,Year,B1ACB,B1PRB,BDACB,BDCCB,BDEIB,BDFDB,BDRCB,CLACB,CLCCB,...,TPOPP,WWCCB,WWEIB,WWICB,WWPRB,WYCCB,WYEGB,WYICB,WYPRB,cluster_id
count,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,...,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000,3315.000000
mean,1992.000000,532.407541,416.246154,917.412368,6.871493,2.733937,918.911011,22.733032,68.404223,2487.257014,...,5093.722775,1349.081448,4605.007843,28716.892006,45616.519759,1.858824,4381.391554,1.108296,4384.358974,1.627451
std,18.764493,11105.879116,5055.926594,3015.018162,90.844368,41.852275,3849.278703,259.759414,458.995586,6031.015416,...,5786.152031,2299.237504,11032.960536,36193.426814,46878.944630,12.122115,21331.967884,13.926656,21340.802962,1.596324
min,1960.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,229.000000,0.000000,-35.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000
25%,1976.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,...,1298.500000,160.000000,0.000000,1872.000000,7340.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1992.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,324.000000,...,3352.000000,443.000000,17.000000,15714.000000,30821.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,2008.000000,0.000000,0.000000,235.500000,0.000000,0.000000,0.000000,0.000000,0.000000,2659.500000,...,6053.500000,1501.000000,2920.000000,43179.000000,69678.500000,0.000000,35.000000,0.000000,35.000000,3.000000
max,2024.000000,426607.000000,137029.000000,37012.000000,2194.000000,900.000000,47215.000000,4668.000000,14588.000000,89986.000000,...,39528.000000,18732.000000,82057.000000,244662.000000,255738.000000,184.000000,423924.000000,344.000000,424035.000000,6.000000


In [4]:
# filter out years from SEDS that don't align with AQI dataset
seds_df = seds_df[seds_df['Year'].between(1980,2024,inclusive='both')]
seds_df.describe()

,Year,B1ACB,B1PRB,BDACB,BDCCB,BDEIB,BDFDB,BDRCB,CLACB,CLCCB,...,TPOPP,WWCCB,WWEIB,WWICB,WWPRB,WYCCB,WYEGB,WYICB,WYPRB,cluster_id
count,2295.000000,2295.000000,2295.000000,2295.000000,2295.000000,2295.00000,2295.000000,2295.000000,2295.0,2295.000000,...,2295.000000,2295.000000,2295.000000,2295.000000,2295.000000,2295.000000,2295.000000,2295.000000,2295.000000,2295.000000
mean,2002.000000,769.033115,601.244444,1325.151198,9.925490,3.94902,1327.315904,32.836601,0.0,1660.196514,...,5588.875817,1868.581699,6626.058388,32670.406536,52751.528976,2.684967,6328.676688,1.600871,6332.962963,1.627451
std,12.990004,13341.679714,6067.712290,3548.477594,109.049708,50.25581,4567.582310,311.680898,0.0,3478.272604,...,6292.131891,2597.773674,12749.567568,40700.932173,51866.268700,14.493591,25397.984553,16.715299,25408.375916,1.596431
min,1980.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.000000,...,405.000000,0.000000,-1.000000,0.000000,24.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000
25%,1991.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.000000,...,1416.500000,322.500000,0.000000,1476.000000,8437.500000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2002.000000,0.000000,0.000000,8.000000,0.000000,0.00000,0.000000,0.000000,0.0,170.000000,...,3760.000000,908.000000,786.000000,16360.000000,33291.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,2013.000000,0.000000,0.000000,903.000000,0.000000,0.00000,61.500000,0.000000,0.0,2008.000000,...,6542.000000,2205.000000,7330.500000,54689.000000,88424.500000,0.000000,1404.000000,0.000000,1404.000000,3.000000
max,2024.000000,426607.000000,137029.000000,37012.000000,2194.000000,900.00000,47215.000000,4668.000000,0.0,35276.000000,...,39528.000000,18732.000000,82057.000000,244662.000000,255738.000000,184.000000,423924.000000,344.000000,424035.000000,6.000000


In [5]:
set_aqi_states = set(aqi_df.State.unique())
set_seds_states = set(seds_df.State.unique())
set_aqi_states - set_seds_states

abbrevs = {
    'AK': 'Alaska',
    'AL': 'Alabama',
    'AR': 'Arkansas',
    'AZ': 'Arizona',
    'CA': 'California',
    'CO': 'Colorado',
    'CT': 'Connecticut',
    'DC': 'District Of Columbia',
    'DE': 'Delaware',
    'FL': 'Florida',
    'GA': 'Georgia',
    'HI': 'Hawaii',
    'IA': 'Iowa',
    'ID': 'Idaho',
    'IL': 'Illinois',
    'IN': 'Indiana',
    'KS': 'Kansas',
    'KY': 'Kentucky',
    'LA': 'Louisiana',
    'MA': 'Massachusetts',
    'MD': 'Maryland',
    'ME': 'Maine',
    'MI': 'Michigan',
    'MN': 'Minnesota',
    'MO': 'Missouri',
    'MS': 'Mississippi',
    'MT': 'Montana',
    'NC': 'North Carolina',
    'ND': 'North Dakota',
    'NE': 'Nebraska',
    'NH': 'New Hampshire',
    'NJ': 'New Jersey',
    'NM': 'New Mexico',
    'NV': 'Nevada',
    'NY': 'New York',
    'OH': 'Ohio',
    'OK': 'Oklahoma',
    'OR': 'Oregon',
    'PA': 'Pennsylvania',
    'RI': 'Rhode Island',
    'SC': 'South Carolina',
    'SD': 'South Dakota',
    'TN': 'Tennessee',
    'TX': 'Texas',
    'UT': 'Utah',
    'VA': 'Virginia',
    'VT': 'Vermont',
    'WA': 'Washington',
    'WI': 'Wisconsin',
    'WV': 'West Virginia',
    'WY': 'Wyoming',
}

In [6]:
len(abbrevs)

51

In [7]:
seds_df['State'] = seds_df.State.replace(abbrevs)

In [8]:
set_seds_states = set(seds_df.State.unique())

There are 9 rows in the SEDS dataset that aren't in the AQI data (South Dakota 1980-4 and Wyoming 1980-3). These will be filtered out.

In [10]:
# checking for data in SEDS that is not in the AQI data (and vice versa)
set_seds_tups = set(zip(seds_df.State,seds_df.Year))
set_aqi_tups = set(zip(aqi_df.State,aqi_df.Year))
set_seds_tups - set_aqi_tups

{('South Dakota', 1980),
 ('South Dakota', 1981),
 ('South Dakota', 1982),
 ('South Dakota', 1983),
 ('South Dakota', 1984),
 ('Wyoming', 1980),
 ('Wyoming', 1981),
 ('Wyoming', 1982),
 ('Wyoming', 1983)}

In [11]:
set_aqi_tups - set_seds_tups

set()

In [12]:
df = aqi_df.merge(seds_df,how='left',on=['State','Year'])
df.head()

,State,Year,Median AQI,B1ACB,B1PRB,BDACB,BDCCB,BDEIB,BDFDB,BDRCB,...,TPOPP,WWCCB,WWEIB,WWICB,WWPRB,WYCCB,WYEGB,WYICB,WYPRB,cluster_id
0,Alabama,1980,47.654124,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3900.0,406.0,0.0,124282.0,141033.0,0.0,0.0,0.0,0.0,0
1,Alabama,1981,45.169604,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3919.0,448.0,0.0,131173.0,150242.0,0.0,0.0,0.0,0.0,0
2,Alabama,1982,41.207882,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3925.0,584.0,0.0,126899.0,153269.0,0.0,0.0,0.0,0.0,0
3,Alabama,1983,44.225619,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3934.0,469.0,0.0,143334.0,164483.0,0.0,0.0,0.0,0.0,0
4,Alabama,1984,44.986195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3952.0,634.0,0.0,146230.0,175126.0,0.0,0.0,0.0,0.0,0


In [13]:
df.describe()

,Year,Median AQI,B1ACB,B1PRB,BDACB,BDCCB,BDEIB,BDFDB,BDRCB,CLACB,...,TPOPP,WWCCB,WWEIB,WWICB,WWPRB,WYCCB,WYEGB,WYICB,WYPRB,cluster_id
count,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.0,...,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000,2286.000000
mean,2002.079615,40.128806,772.060805,603.611549,1330.368329,9.964567,3.964567,1332.541557,32.965879,0.0,...,5608.497813,1875.725722,6652.145232,32795.004374,52945.841645,2.695538,6353.590989,1.607174,6357.894138,1.624672
std,12.953029,7.197556,13367.841065,6079.532529,3554.482351,109.262473,50.354073,4575.807546,312.287282,0.0,...,6296.713420,2600.382236,12767.854567,40732.417616,51875.549565,14.521125,25444.841799,16.747883,25455.250665,1.596217
min,1980.000000,13.881630,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,405.000000,0.000000,-1.000000,0.000000,24.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000
25%,1991.000000,35.657875,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,1435.000000,325.250000,0.000000,1527.000000,8578.500000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2002.000000,40.193498,0.000000,0.000000,9.500000,0.000000,0.000000,0.000000,0.000000,0.0,...,3792.500000,927.000000,796.000000,16595.500000,33488.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,2013.000000,44.411370,0.000000,0.000000,928.250000,0.000000,0.000000,69.500000,0.000000,0.0,...,6560.500000,2212.000000,7365.500000,54751.500000,88870.250000,0.000000,1416.750000,0.000000,1416.750000,3.000000
max,2024.000000,71.000000,426607.000000,137029.000000,37012.000000,2194.000000,900.000000,47215.000000,4668.000000,0.0,...,39528.000000,18732.000000,82057.000000,244662.000000,255738.000000,184.000000,423924.000000,344.000000,424035.000000,6.000000


# Pre-processing -- normalizing energy features by population and log transforming energy features

In [15]:
cols = list(df.columns)
cols.remove('Median AQI')

categorical_features = ['State', 'cluster_id']
pop_col = 'TPOPP'
year_col = 'Year'
numeric_features = [col for col in cols if (col not in categorical_features) and (col != pop_col)]

# normalize energy features
for col in [col for col in numeric_features if col not in year_col]:
    # normalize energy features by total population
    df[col] = df[col] / df[pop_col]
    # log transform energy features
    df[col] = df[col].apply(np.log1p)

df = df.drop(columns=pop_col)

# Creating pipelines

In [17]:
def run_reg_pipeline(cluster:bool, regularizer:str, X_train:pd.DataFrame,y_train:pd.Series):
    numeric_transformer = Pipeline(steps=[('scaler',StandardScaler())])

    if not cluster:
        categorical_features = ['State']
        X_train_copy = X_train.copy().drop(columns='cluster_id')
    else:
        categorical_features = ['State','cluster_id']
        X_train_copy = X_train
        
    categorical_transformer = Pipeline(steps = [('enc',OneHotEncoder())])
    
    preprocessor = ColumnTransformer(
        transformers = [
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    if regularizer == 'ridge':
        RidgePipe = Pipeline(steps=[('preprocessor',preprocessor),
        ('ridge',RidgeCV(alphas=np.linspace(0.1,20,1000)))])

        return RidgePipe.fit(X_train_copy, y_train)
    
    elif regularizer == 'lasso':
        LassoPipe = Pipeline(steps=[('preprocessor',preprocessor),
        ('lasso',LassoCV(cv = 5, random_state=0, n_jobs=-1, verbose=True,max_iter=100000))])

        return LassoPipe.fit(X_train_copy, y_train)

    elif regularizer == 'elasticnet':
        ElasticNetPipe = Pipeline(steps=[('preprocessor',preprocessor),
        ('elastic',ElasticNetCV(cv = None, random_state=0, n_jobs=-1, verbose=True,max_iter=100000))])

        return ElasticNetPipe.fit(X_train_copy, y_train)
        

In [18]:
# hold out data after 2020 for test set
df_train = df[df['Year'] < 2020]
df_test = df[df['Year'] >= 2020]
y_train = df_train.pop('Median AQI')
y_test = df_test.pop('Median AQI')
X_train = df_train
X_test = df_test

In [19]:
cluster_ridgereg = run_reg_pipeline(cluster=True, regularizer='ridge', X_train=X_train,y_train=y_train)
no_cluster_ridgereg = run_reg_pipeline(cluster=False, regularizer='ridge', X_train=X_train,y_train=y_train)

cluster_lassoreg = run_reg_pipeline(cluster=True, regularizer='lasso', X_train=X_train,y_train=y_train)
no_cluster_lassoreg = run_reg_pipeline(cluster=False, regularizer='lasso', X_train=X_train,y_train=y_train)

cluster_elasticreg = run_reg_pipeline(cluster=True, regularizer='elasticnet', X_train=X_train,y_train=y_train)
no_cluster_elasticreg = run_reg_pipeline(cluster=False, regularizer='elasticnet', X_train=X_train,y_train=y_train)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
....................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.5s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
.........................................................................................................................................................................................................................................................................

In [20]:
def score_dataset(model,X_data=X_test, y_data=y_test):    
    y_pred = model.predict(X_data)
    return {'rmse':root_mean_squared_error(y_data, y_pred),'r2':r2_score(y_data,y_pred)}

In [21]:
models = {'ridge cluster':cluster_ridgereg, 
          'ridge no cluster':no_cluster_ridgereg,
          'lasso cluster':cluster_lassoreg,
          'lasso no cluster':no_cluster_lassoreg,
         'elasticnet cluster':cluster_elasticreg,
         'elasticnet no cluster':no_cluster_elasticreg}
for name, model in models.items():
    test_scores = score_dataset(model)
    print(f"Model is {name}. Test RMSE score: {test_scores['rmse']}. Test R2 score: {test_scores['r2']}")
    
    train_scores = score_dataset(model,X_data = X_train, y_data = y_train)
    print(f"Model is {name}. Train RMSE score: {train_scores['rmse']}. Train R2 score: {train_scores['r2']}\n")

Model is ridge cluster. Test RMSE score: 4.529482487464749. Test R2 score: 0.2749200188742985
Model is ridge cluster. Train RMSE score: 3.902756238213018. Train R2 score: 0.721517428862648

Model is ridge no cluster. Test RMSE score: 4.566521456574536. Test R2 score: 0.26301313067857857
Model is ridge no cluster. Train RMSE score: 3.901628268289095. Train R2 score: 0.7216783789974357

Model is lasso cluster. Test RMSE score: 4.98624505538583. Test R2 score: 0.12130939619272885
Model is lasso cluster. Train RMSE score: 7.048721625081791. Train R2 score: 0.09160259378742286

Model is lasso no cluster. Test RMSE score: 4.98624505538583. Test R2 score: 0.12130939619272885
Model is lasso no cluster. Train RMSE score: 7.048721625081791. Train R2 score: 0.09160259378742286

Model is elasticnet cluster. Test RMSE score: 4.8754296498749765. Test R2 score: 0.15993182292383068
Model is elasticnet cluster. Train RMSE score: 6.77033075552745. Train R2 score: 0.16194033497583438

Model is elasticnet